In [1]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
ee.Initialize(project="ee-sentinel-ts-an-detection")

In [23]:
# Output Folder in Google Drive
folder="python_wrapper"

# 1. AOI laden
aoi = ee.FeatureCollection(
    "projects/ee-sentinel-ts-an-detection/assets/test_area_small"
)

# 2. Parameter setzen
start = "2025-01-01"
end = "2026-01-15"

# Kernal für die bild Verarbeitungsvariablen
kernel = ee.Kernel.square(radius=1, units="pixels")
#kernel2 = ee.Kernel.square(radius=2, units="pixels")


band_list = [
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B11",
    "B12",
    "SCL",
    "QA60",
]

# Bänder, für die die 3x3-Varianz berechnet werden soll (ohne SCL/QA60)
var_bands = [
 #   "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8"
  #  "B8A",
  #  "B11",
  #  "B12",
]




# Sentinel-2-Collection
S2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(start, end)
    .select(band_list)
)


## Funktionen


In [24]:

# Für jedes Bild: 3x3-Varianz für var_bands berechnen und als Bänder anhängen
def add_neighborhood_variance(image, kernel_select,var_bands):
    # Neighborhood-Reduktion (Varianz) für die Spektralbänder
    nbr_var = (
        image.select(var_bands)
        .reduceNeighborhood(
            reducer=ee.Reducer.variance(),
            kernel=kernel_select,
#            scale=10,
            skipMasked=True,
        )
    )

    # Bänder umbenennen: B2 -> B2_var etc.
    var_band_names = nbr_var.bandNames().map(
        lambda b: ee.String(b).cat("_var")
    )
    nbr_var = nbr_var.rename(var_band_names)

    # Varianz-Bänder an Originalbild anhängen
    return image.addBands(nbr_var)

def add_neighborhood_mean(image, kernel_select,mean_bands):
    # Neighborhood-Reduktion (Varianz) für die Spektralbänder
    nbr_mean = (
        image.select(mean_bands)
        .reduceNeighborhood(
            reducer=ee.Reducer.mean(),
            kernel=kernel_select,
#            scale=10,
            skipMasked=True,
        )
    )

    # Bänder umbenennen: B2 -> B2_var etc.
    mean_band_names = nbr_mean.bandNames().map(
        lambda b: ee.String(b).cat("_mean")
    )
    nbr_mean = nbr_mean.rename(mean_band_names)

    # Varianz-Bänder an Originalbild anhängen
    return image.addBands(nbr_mean)

def month_ranges(start, end):
    return pd.period_range(start=start, end=end, freq="M")



# Für jedes Bild samplen und FeatureCollection sammeln
def sample_image(image):
    time_str = image.date().format("YYYY-MM-dd")
    sampled = image.sample(
        region=aoi,
        scale=10,
        projection="EPSG:4326",
        # numPixels=3000,  # bei Bedarf wieder aktivieren
        geometries=True,
    )
    # Nur SCL 4,5 (Vegetation) behalten
    filtered = sampled.filter(ee.Filter.inList("SCL", [4, 5]))
    # Zeitstempel als Property setzen
    return filtered.map(lambda f: f.set("time", time_str))

# In the function definitions cell, add this new function:

def sample_points(image, point_aoi):
    """Sample image at point locations using sampleRegions, no SCL filter."""
    time_str = image.date().format("YYYY-MM-dd")
    sampled = image.sampleRegions(
        collection=point_aoi,
        scale=10,
        projection="EPSG:4326",
        geometries=True,
    )
    return sampled.map(lambda f: f.set("time", time_str))

# In the point AOI processing cell, update the data_collection line:

def export_monthly(fc, period, folder):
    start = period.to_timestamp("M") - pd.offsets.MonthEnd(0)
    start = start.replace(day=1)
    end = period.to_timestamp("M")

    start_str = start.strftime("%Y-%m-%d")
    end_str = end.strftime("%Y-%m-%d")

    subset = fc.filter(
        ee.Filter.And(
            ee.Filter.greaterThanOrEquals("time", start_str),
            ee.Filter.lessThanOrEquals("time", end_str)
        )
    )

    task = ee.batch.Export.table.toDrive(
        collection=subset,
        description=f"samples_{start_str}",
        folder=folder,
        fileNamePrefix=f"samples_{start_str}",
        fileFormat="CSV",
    )
    task.start()

    print(f"Export gestartet: {start_str} – {end_str}")


In [25]:
S2_with_var = S2.map(
    lambda img: add_neighborhood_variance(
        img,
        kernel_select=kernel,
        var_bands=var_bands
    )
)
S2_with_var = S2_with_var.map(
    lambda img: add_neighborhood_mean(
        img,
        kernel_select=kernel,
        mean_bands=var_bands
    )
)



data_collection = S2_with_var.map(sample_image).flatten()



In [ ]:
months = month_ranges(start, end)

for p in months:
    export_monthly(data_collection, p, folder=folder)


# New cell for point AOI processing


In [29]:

# # Initialize if not already done
# ee.Initialize(project="ee-sentinel-ts-an-detection")

# Load point AOI (replace with your asset path)
point_aoi = ee.FeatureCollection("projects/ee-sentinel-ts-an-detection/assets/conifer_disturbed").limit(200)
# Set parameters (adjust as needed)
start = "2020-08-01"
end = "2021-01-15"
folder = "test_point_aoi_exports"

# Kernel for neighborhood stats
kernel = ee.Kernel.square(radius=1, units="pixels")

# Band lists (same as before)
band_list = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12", "SCL", "QA60"]
var_bands = ["B3", "B4", "B5", "B6", "B7", "B8"]




In [30]:
# Sentinel-2 Collection filtered to point AOI bounds
S2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(point_aoi)
    .filterDate(start, end)
    .select(band_list)
)

# Add neighborhood variance and mean
S2_with_stats = S2.map(
    lambda img: add_neighborhood_variance(img, kernel_select=kernel, var_bands=var_bands)
).map(
    lambda img: add_neighborhood_mean(img, kernel_select=kernel, mean_bands=var_bands)
)

# Sample at points (no buffer)
data_collection = S2_with_stats.map(lambda img: sample_points(img, point_aoi)).flatten()
# Export monthly
months = month_ranges(start, end)
for p in months:
    # In der export_monthly-Funktion, vor dem task.start():
    export_monthly(data_collection, p, folder=folder)
    print(f"Export gestartet für Monat: {p} in folder: {folder} ")

Export gestartet: 2020-08-01 – 2020-08-31
Export gestartet für Monat: 2020-08 in folder: test_point_aoi_exports 
Export gestartet: 2020-09-01 – 2020-09-30
Export gestartet für Monat: 2020-09 in folder: test_point_aoi_exports 
Export gestartet: 2020-10-01 – 2020-10-31
Export gestartet für Monat: 2020-10 in folder: test_point_aoi_exports 
Export gestartet: 2020-11-01 – 2020-11-30
Export gestartet für Monat: 2020-11 in folder: test_point_aoi_exports 
Export gestartet: 2020-12-01 – 2020-12-31
Export gestartet für Monat: 2020-12 in folder: test_point_aoi_exports 
Export gestartet: 2021-01-01 – 2021-01-31
Export gestartet für Monat: 2021-01 in folder: test_point_aoi_exports 


In [16]:
data_collection